# Lab Assignment - Wisconsin Breast Cancer
## JS03 - Feature Extraction

**Name:** Tiara Febrianie  
**Student ID:** 244107020097

This assignment applies encoding, standardization, feature selection, and Logistic Regression to the Wisconsin Breast Cancer dataset.

## Assignment Objectives

1. Identify usable and unusable variables.
2. Encode the `diagnosis` target column.
3. Standardize numerical feature columns.
4. Select the optimal number of features with `SelectKBest`.
5. Test the selected features with Logistic Regression.

## Step 0 - Load the Libraries

In [1]:
# Uncomment this line if the packages are not installed yet.
# %pip install -q pandas scikit-learn

import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Step 1 - Load and Inspect the Dataset

The dataset contains 569 observations, one identifier column, one diagnosis column, and 30 numerical measurement columns.

In [2]:
df = pd.read_csv('wbc.csv')

print('Dataset shape:', df.shape)
print('Missing values:', df.isna().sum().sum())
df.head()

Dataset shape: (569, 33)
Missing values: 569


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


## Step 2 - Identify Usable and Unusable Variables

The `id` column is not a medical measurement, so it is excluded. The `diagnosis` column is the target and is not used as an input feature. The remaining numerical columns are usable model features.

In [3]:
id_column = 'id'
target_column = 'diagnosis'
empty_columns = [column for column in df.columns if df[column].isna().all()]
feature_columns = [
    column for column in df.columns
    if column not in [id_column, target_column] + empty_columns
]

print('Unusable identifier column:', id_column)
print('Unusable empty columns:', empty_columns)
print('Target column:', target_column)
print('Number of usable features:', len(feature_columns))
print('Usable features:')
print(feature_columns)

Unusable identifier column: id
Unusable empty columns: ['Unnamed: 32']
Target column: diagnosis
Number of usable features: 30
Usable features:
['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']


## Step 3 - Encode the Diagnosis Column

`B` (Benign) is encoded as 0 and `M` (Malignant) is encoded as 1.

In [4]:
diagnosis_mapping = {'B': 0, 'M': 1}
y = df[target_column].map(diagnosis_mapping)
X = df[feature_columns].copy()

if y.isna().any():
    raise ValueError('The diagnosis column contains an unknown label.')

print('Diagnosis mapping:', diagnosis_mapping)
print(y.value_counts().sort_index())

Diagnosis mapping: {'B': 0, 'M': 1}
diagnosis
0    357
1    212
Name: count, dtype: int64


## Step 4 - Split the Data

A stratified split preserves the proportion of benign and malignant cases in the training and testing sets.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

Training samples: 455
Testing samples: 114


## Step 5 - Select the Optimal Number of Features

The pipeline standardizes each feature using only the training folds, selects the top `k` features with ANOVA F-test, and trains Logistic Regression. Grid search compares every possible value of `k` using 5-fold stratified cross-validation.

In [6]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif)),
    ('model', LogisticRegression(max_iter=5000, random_state=42))
])

parameter_grid = {
    'selector__k': list(range(1, len(feature_columns) + 1))
}
cross_validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=parameter_grid,
    cv=cross_validation,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print('Optimal number of features:', grid_search.best_params_['selector__k'])
print('Best mean cross-validation accuracy:', round(grid_search.best_score_, 4))

Optimal number of features: 29
Best mean cross-validation accuracy: 0.9736


In [7]:
cv_results = pd.DataFrame(grid_search.cv_results_)
feature_count_results = cv_results[[
    'param_selector__k',
    'mean_test_score',
    'std_test_score'
]].sort_values('mean_test_score', ascending=False)
feature_count_results.head(10)

,param_selector__k,mean_test_score,std_test_score
28,29,0.973626,0.014906
29,30,0.973626,0.014906
23,24,0.973626,0.017855
24,25,0.973626,0.017855
25,26,0.973626,0.017855
27,28,0.971429,0.017855
26,27,0.971429,0.017855
20,21,0.969231,0.012815
22,23,0.969231,0.017582
19,20,0.969231,0.012815


## Step 6 - Display the Selected Features

The selector is fitted only on the training data. Its support mask identifies the measurement columns retained by the best model.

In [8]:
best_selector = grid_search.best_estimator_.named_steps['selector']
selected_features = [
    feature for feature, selected in zip(feature_columns, best_selector.get_support())
    if selected
]

print('Number of selected features:', len(selected_features))
print('Selected features:')
print(selected_features)

Number of selected features: 29
Selected features:
['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']


## Step 7 - Test the Best Pipeline

The final evaluation is performed on the test set, which was not used during feature selection.

In [9]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print('Test accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('Classification report:')
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))

Test accuracy: 0.9649
Confusion matrix:
[[71  1]
 [ 3 39]]
Classification report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.97        72
   Malignant       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



## Conclusion

The identifier `id` was excluded, while `diagnosis` was encoded as the target. Standardization, feature selection, and Logistic Regression were combined in one pipeline to avoid data leakage. The optimal number of features and their names are printed by Steps 5 and 6, and the final model performance is reported in Step 7.